# Week 13 — Block 2: Guided Demo (ML/NLP Visualization)

**DATS 6401 · Visualization of Complex Data**

~35 min: model-explanation visuals, then **connect the Streamlit client to the Week 12 server** and visualize a *live* prediction.

Parts 1–2 run offline in this notebook. Part 3 needs the Week 12 server running: `uvicorn server_solution:app --reload` (from the week12 folder).

## Part 1 — Performance visuals (~12 min)

Same model the server wraps; here we look *inside* it.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

iris = load_iris(as_frame=True)
X, y = iris.data, iris.target
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42)
model = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_tr, y_tr)
print("Accuracy:", round(model.score(X_te, y_te), 3))

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_estimator(model, X_te, y_te,
                                      display_labels=iris.target_names)
plt.title("Confusion matrix — WHERE does the model err, not just how often")
plt.show()

In [ ]:
# ROC needs a BINARY problem (Week 4 of the 2102 book hit this same bug!):
# reduce to versicolor vs virginica.
from sklearn.metrics import RocCurveDisplay

mask = y.isin([1, 2])
Xb, yb = X[mask], (y[mask] == 2).astype(int)
Xb_tr, Xb_te, yb_tr, yb_te = train_test_split(Xb, yb, test_size=0.3, random_state=42)
bmodel = RandomForestClassifier(random_state=42).fit(Xb_tr, yb_tr)

RocCurveDisplay.from_estimator(bmodel, Xb_te, yb_te)
plt.title("ROC — versicolor vs virginica")
plt.show()

## Part 2 — Behavior visuals (~10 min)

In [ ]:
import pandas as pd

importances = pd.Series(model.feature_importances_, index=X.columns).sort_values()
importances.plot.barh(title="Feature importance — the petal measurements do the work")
plt.show()

In [ ]:
# Embedding view (PCA here; swap in UMAP if umap-learn is installed)
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

pcs = PCA(n_components=2).fit_transform(StandardScaler().fit_transform(X))
preds = model.predict(X)
correct = preds == y

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(pcs[correct, 0], pcs[correct, 1], c=y[correct], cmap="Set2", s=30, label="correct")
ax.scatter(pcs[~correct, 0], pcs[~correct, 1], c="red", marker="x", s=80, label="wrong")
ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
ax.legend(); ax.set_title("Where in feature space does the model fail? (red ✗)")
plt.show()

**Narrate:** the errors cluster on the versicolor/virginica boundary — the *embedding shows you where the model is unsure*, which no accuracy number can.

## Part 3 — The live wire (~13 min)

1. Terminal 1: `uvicorn server_solution:app --reload` (week12 folder)
2. Run the next cell: this notebook becomes a *client*.
3. Then: `streamlit run app_live_predictions.py` — the same call with sliders and a confidence bar. **Kill the server mid-demo** to show the graceful-failure path.

In [ ]:
import requests

payload = {"sepal_length": 6.1, "sepal_width": 2.8, "petal_length": 4.7, "petal_width": 1.2}
try:
    r = requests.post("http://127.0.0.1:8000/predict", json=payload, timeout=5)
    r.raise_for_status()
    out = r.json()
    print("LIVE prediction:", out["prediction"], f"({out['confidence']:.0%} confident)")
    print("Full distribution:", out["probabilities"])
except requests.RequestException as e:
    print("Server not reachable:", e)
    print("-> start it:  uvicorn server_solution:app --reload  (week12 folder)")